## 문서 파싱 실습

`sandbox/w5/day02/documents/` 에 받아 둔 것을 쓴다.

할 일
1. 문서를 전부 파싱한다 (5건 이상)
2. 문서마다 `ParsedDoc` 의 모양 — 문단 수 · 표 수 · 문단 앞 50글자
3. 표를 마크다운으로 모아 파일 하나로 저장한다 — **PDF 의 표까지**

In [1]:
import logging
import os
import pathlib
import sys
import time

# 노트북이 어느 위치에서 실행되든 backend/app 이 들어 있는 폴더를 찾아 루트로 삼는다
here = pathlib.Path.cwd().resolve()
candidates = [here, *here.parents, here / "hanwha-agent"]
ROOT = next((p for p in candidates if (p / "backend" / "app").is_dir()), None)
if ROOT is None:
    raise RuntimeError(f"hanwha-agent 루트를 찾지 못했습니다. 현재 위치: {here}")

os.chdir(ROOT)
BACKEND = str(ROOT / "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

SANDBOX = ROOT / "sandbox" / "w5" / "day02"
DOCUMENTS = SANDBOX / "documents"

print("프로젝트 루트 :", ROOT)
print("문서 폴더     :", DOCUMENTS.relative_to(ROOT))

프로젝트 루트 : C:\workspace\hanwha-agent
문서 폴더     : sandbox\w5\day02\documents


### 받아 둔 문서

In [3]:
파일들 = sorted(p for p in DOCUMENTS.rglob("*") if p.is_file())

if not 파일들:
    print(f"문서가 없습니다 → {DOCUMENTS.relative_to(ROOT)} 에 넣고 다시 실행하세요.")
else:
    print(f"{'파일':<52}{'크기':>10}")
    print("─" * 80)
    for p in 파일들:
        print(f"{p.name[:50]:<52}{p.stat().st_size / 1024:>8,.0f} KB")
    print("─" * 80)
    print(f"{len(파일들)}건")

파일                                                          크기
────────────────────────────────────────────────────────────────────────────────
GNSM_20210831_교육문화과_수업지도안_정보보안전문가.pdf                    103 KB
TTPs_2_스피어_피싱으로_정보를_수집하는_공격망_구성_방식.pdf                10,303 KB
개인정보 보호법(법률)(제21445호)(20260911).hwpx                     459 KB
글로벌 AI 보안 동향 조사·분석 보고서(2026)_6호.pdf                      453 KB
도로교통법(법률)(제21246호)(20260701).hwpx                        681 KB
────────────────────────────────────────────────────────────────────────────────
5건


### 1. 전부 파싱한다

진입점 `parse_local` 하나로 부른다. 확장자를 보고 알아서 파서를 고른다.

In [ ]:
from app.core.exceptions import ValidationFailed
from app.rag.local_parsers import parse_local

logging.disable(logging.WARNING)

결과 = {}       
실패 = []

for path in 파일들:
    t0 = time.perf_counter()
    try:
        doc = parse_local(path)
    except ValidationFailed as exc:
        실패.append((path.name, str(exc)))
        continue
    if doc is None:
        실패.append((path.name, "파서가 터졌습니다 (로그 확인)"))
        continue
    결과[path.name] = (doc, time.perf_counter() - t0)

logging.disable(logging.NOTSET)

print(f"읽음 {len(결과)}건 · 실패 {len(실패)}건")
for 이름, 사유 in 실패:
    print(f"  실패  {이름[:44]:<46} {사유}")

읽음 5건 · 실패 0건


### 2. 문서마다 무엇이 나왔나

In [4]:
for 이름, (doc, 초) in 결과.items():
    조항 = [b for b in doc.blocks if b.kind == "조항"]
    표 = [b for b in doc.blocks if b.kind == "표"]

    print("━" * 78)
    print(이름)
    print(f"  ParsedDoc(blocks={len(doc.blocks)}, page_count={doc.page_count}, "
          f"table_count={doc.table_count})")
    print(f"  문단 {len(조항)}개 · 표 {len(표)}개 · 파싱 {초:.1f}초")
    print("  문단 앞 50글자")
    for b in 조항[:3]:
        print(f"    {b.text[:50]}")
    if not 조항:
        print("    (문단 없음)")
print("━" * 78)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GNSM_20210831_교육문화과_수업지도안_정보보안전문가.pdf
  ParsedDoc(blocks=1, page_count=1, table_count=0)
  문단 1개 · 표 0개 · 파싱 0.2초
  문단 앞 50글자
    - 1 -
수업 지도안    정보보안전문가
과정   SW 융합과정 대상 중등 전시관
학습 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TTPs_2_스피어_피싱으로_정보를_수집하는_공격망_구성_방식.pdf
  ParsedDoc(blocks=78, page_count=79, table_count=0)
  문단 78개 · 표 0개 · 파싱 6.8초
  문단 앞 50글자
    집        필  침해사고분석단    종합분석팀: 
김동욱    선임 김병재    선임
    한  국  인  터  넷  진  흥  원     1
서론1. 
해킹  사고가  지속  발생
    한  국  인  터  넷  진  흥  원     2
2.  개요
지난  월에  공개된 4 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
개인정보 보호법(법률)(제21445호)(20260911).hwpx
  ParsedDoc(blocks=1151, page_count=1, table_count=1)
  문단 1150개 · 표 1개 · 파싱 0.1초
  문단 앞 50글자
    개인정보 보호법
    개인정보보호위원회(심사총괄담당관 - 일반 법령해석) 02-2100-3043
    개인정보보호위원회(국제협력담당관 - 국외이전) 02-2100-2484, 2499
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
# 형식별로 묶어서 견줘 본다
from collections import defaultdict

형식별 = defaultdict(list)
for 이름, (doc, 초) in 결과.items():
    형식별[pathlib.Path(이름).suffix.lower()].append((이름, doc))

print(f"{'형식':<8}{'문서':>4}{'블록 합':>9}{'표 합':>7}{'블록/문서':>11}")
print("─" * 42)
for ext, 목록 in sorted(형식별.items()):
    블록 = sum(len(d.blocks) for _, d in 목록)
    표 = sum(d.table_count for _, d in 목록)
    print(f"{ext:<8}{len(목록):>4}{블록:>9}{표:>7}{블록 // len(목록):>11}")

형식        문서     블록 합    표 합      블록/문서
──────────────────────────────────────────
.hwpx      2     2861      2       1430
.pdf       3       92      0         30


문서마다 성격이 다르다.

- **법령(hwpx)** — 조항이 천 개 단위다. 조문 하나하나가 문단으로 잘려 있다
- **보고서(pdf)** — 쪽이 곧 블록이라 열 몇 개다
- **PDF 의 표가 0 이다** — `parse_pdf` 는 표를 찾지 않는다

### 3. 파서가 잡은 표를 본다

개수만 세면 안 된다. **행이 몇 개인지** 본다.

In [6]:
print(f"{'문서':<44}{'표':<7}{'행':>4}{'열':>4}  첫 줄")
print("─" * 84)
있음 = False
for 이름, (doc, _) in 결과.items():
    for b in (x for x in doc.blocks if x.kind == "표"):
        있음 = True
        줄 = b.text.splitlines()
        print(f"{이름[:42]:<44}{b.locator:<7}{len(줄) - 2:>4}{줄[0].count('|') - 1:>4}  {줄[0][:26]}")
if not 있음:
    print("  (파서가 잡은 표 없음)")

문서                                          표         행   열  첫 줄
────────────────────────────────────────────────────────────────────────────────────
개인정보 보호법(법률)(제21445호)(20260911).hwpx        표1        0   1  | 개인정보 보호법 |
도로교통법(법률)(제21246호)(20260701).hwpx           표1        0   1  | 도로교통법 |


**둘 다 본문 행이 0 이다.** 머리 행 하나짜리 — 법령 맨 위의 제목 상자다. 표처럼 그려져 있을 뿐 담긴 자료가 없다.

법령은 조문이 전부 글이라 표를 거의 쓰지 않는다. 그렇다면 쓸 만한 표는 PDF 쪽 보고서에 있을 텐데, `parse_pdf` 는 그걸 못 본다.

### 4. PDF 의 표는 따로 꺼낸다

`pdfplumber` 는 **선을 보고** 표를 찾는다.

In [7]:
import pdfplumber

pdf표 = {}        # 파일 이름 -> [(쪽, 표) ...]
logging.disable(logging.WARNING)

for path in (p for p in 파일들 if p.suffix.lower() == ".pdf"):
    t0 = time.perf_counter()
    찾음 = []
    with pdfplumber.open(str(path)) as pdf:
        for n, page in enumerate(pdf.pages, 1):
            for t in page.extract_tables():
                찾음.append((n, t))
    pdf표[path.name] = 찾음
    print(f"{path.name[:44]:<46} 표 {len(찾음):>3}개  {time.perf_counter() - t0:>5.1f}초")

logging.disable(logging.NOTSET)
print(f"\nPDF 표 합계 {sum(len(v) for v in pdf표.values())}개")

GNSM_20210831_교육문화과_수업지도안_정보보안전문가.pdf          표   3개    0.2초
TTPs_2_스피어_피싱으로_정보를_수집하는_공격망_구성_방식.pdf         표  13개   13.8초
글로벌 AI 보안 동향 조사·분석 보고서(2026)_6호.pdf            표  17개    1.3초

PDF 표 합계 33개


찾긴 찾는데 **시간이 든다.** 79쪽짜리 하나에 십수 초다. 문서를 올릴 때 한 번 도는 일이라 견딜 만하지만, 문서가 쌓이면 배치로 돌릴 일이다.

### 셀 두 가지를 먼저 손본다

어제 겪은 것이 그대로 나온다.

In [8]:
칸 = [c for v in pdf표.values() for _, t in v for row in t for c in row]
print(f"칸 {len(칸)}개")
print(f"  None 인 칸        {sum(1 for c in 칸 if c is None):>4}개   ← 병합된 자리")
print(f"  줄바꿈이 든 칸    {sum(1 for c in 칸 if c and chr(10) in c):>4}개   ← 마크다운 행을 갈라버린다")


def 셀정리(표) -> list[list[str]]:
    """None 은 빈 문자열로, 셀 안 줄바꿈은 공백으로."""
    return [[(c or "").replace("\n", " ").strip() for c in row] for row in 표]

칸 339개
  None 인 칸          57개   ← 병합된 자리
  줄바꿈이 든 칸      56개   ← 마크다운 행을 갈라버린다


### 다 표는 아니다

찾은 것을 그대로 쓰면 안 된다. **머리글·바닥글 영역이 선으로 그려져 있으면 표로 잡힌다.**

In [9]:
from app.rag.local_parsers import _table_to_markdown

이름, 목록 = next(iter(pdf표.items()))
for 쪽, 표 in 목록[:3]:
    rows = 셀정리(표)
    print(f"── p.{쪽}  {len(rows)}행 {len(rows[0])}열")
    for l in _table_to_markdown(rows[0], rows[1:]).splitlines()[:3]:
        print("   ", l[:92])
    print()

── p.1  1행 2열
    | 수업 지도안 | 정보보안전문가 |
    |---|---|

── p.1  6행 7열
    | 과정 | SW 융합과정 |  | 대상 | 중등 |  | 전시관 |
    |---|---|---|---|---|---|---|
    | 학습 주제 | 보안기술 탐색과 디지털포렌식 실습 |  | 학습 목표 |  |  |  |

── p.1  6행 4열
    | 단계 | 학습 과정 | 교수 - 학습 활동 | 학습자료 및 유의점 |
    |---|---|---|---|
    | 호기심 키우기 | 학습 목표 제시 학습 동기 유발 | ∘ 포렌식(forensic)은 범죄를 밝혀내기 위해 수사에 쓰이는 과학적 수단이나 방법, 기술 등을 말하며 



In [10]:
def 빈칸비율(rows) -> float:
    칸 = [c for r in rows for c in r]
    return sum(1 for c in 칸 if not c) / len(칸) if 칸 else 1.0


def 쓸만한가(rows) -> bool:
    """머리 + 본문 두 줄 이상이고, 열이 둘 이상이고, 빈 칸이 절반 아래."""
    return len(rows) >= 3 and len(rows[0]) >= 2 and 빈칸비율(rows) <= 0.4


print(f"{'문서':<46}{'전체':>5}{'남김':>5}{'버림':>5}")
print("─" * 62)
거른표 = {}
for 이름, 목록 in pdf표.items():
    남김 = [(쪽, 셀정리(t)) for 쪽, t in 목록 if 쓸만한가(셀정리(t))]
    거른표[이름] = 남김
    print(f"{이름[:44]:<46}{len(목록):>5}{len(남김):>5}{len(목록) - len(남김):>5}")

print()
print("남긴 표의 머리 행")
for 이름, 남김 in 거른표.items():
    for 쪽, rows in 남김:
        print(f"  p.{쪽:<4} {' | '.join(rows[0])[:66]}")

문서                                               전체   남김   버림
──────────────────────────────────────────────────────────────
GNSM_20210831_교육문화과_수업지도안_정보보안전문가.pdf             3    1    2
TTPs_2_스피어_피싱으로_정보를_수집하는_공격망_구성_방식.pdf           13    3   10
글로벌 AI 보안 동향 조사·분석 보고서(2026)_6호.pdf              17    5   12

남긴 표의 머리 행
  p.1    단계 | 학습 과정 | 교수 - 학습 활동 | 학습자료 및 유의점
  p.44   스피어 피싱 | 워터링 홀 사이트 유도
  p.48   perf91nc.inf | C2 List
  p.61   연결된 드라이브 정보 수집 | 0x97863654
  p.3    시점 | 핵심 조치 | 세부 실행 내용
  p.5    분야 | 주요 참여 기업
  p.6    방어 영역 | 공개 기술·기여 | 보안 기능
  p.9    통제 대상 | 관리·통제 초점 | 강제·대응 방식
  p.12   가이던스 주요 항목 | 기업 관점의 준비 포인트


**33개가 9개로 줄었다.** 남은 것들은 머리 행만 봐도 무슨 표인지 안다.

기준 셋은 임의로 정한 것이고 문서 종류에 따라 달라진다. 중요한 것은 **찾은 것을 그대로 믿지 않는다**는 쪽이다. 표를 33개 뽑았다고 보고하면 24개가 비어 있을 수 있다.

### 5. 모아서 마크다운 파일로 저장한다

파서가 잡은 표(hwpx·docx)와 `pdfplumber` 로 꺼낸 표(pdf)를 한 파일에 담는다.

In [11]:
조각 = []
셈 = {"파서": 0, "pdfplumber": 0}

for 이름, (doc, _) in 결과.items():
    표블록 = [b for b in doc.blocks if b.kind == "표"]
    남김 = 거른표.get(이름, [])
    if not 표블록 and not 남김:
        continue

    조각.append(f"# {이름}\n")
    for b in 표블록:
        조각.append(f"## {b.locator}  (파서)\n\n{b.text}\n")
        셈["파서"] += 1
    for 쪽, rows in 남김:
        md = _table_to_markdown(rows[0], rows[1:])
        조각.append(f"## p.{쪽}  (pdfplumber)\n\n{md}\n")
        셈["pdfplumber"] += 1

본문 = "\n".join(조각)
OUT = SANDBOX / "documents_tables.md"
OUT.write_text(본문, encoding="utf-8")

print(f"저장 : {OUT.relative_to(ROOT).as_posix()}")
print(f"표 {sum(셈.values())}개 (파서 {셈['파서']} · pdfplumber {셈['pdfplumber']})")
print(f"{len(본문.splitlines())}줄 · {OUT.stat().st_size:,} 바이트")

저장 : sandbox/w5/day02/documents_tables.md
표 11개 (파서 2 · pdfplumber 9)
119줄 · 8,336 바이트


In [12]:
# 저장한 것을 다시 읽어 확인한다
다시 = OUT.read_text(encoding="utf-8")
print("\n".join(다시.splitlines()[:14]))
print("   …")
깨짐 = [l for l in 다시.splitlines() if l.startswith("|") and l.count("|") < 2]
print(f"\n칸이 모자란 줄 : {len(깨짐)}개")

# GNSM_20210831_교육문화과_수업지도안_정보보안전문가.pdf

## p.1  (pdfplumber)

| 단계 | 학습 과정 | 교수 - 학습 활동 | 학습자료 및 유의점 |
|---|---|---|---|
| 호기심 키우기 | 학습 목표 제시 학습 동기 유발 | ∘ 포렌식(forensic)은 범죄를 밝혀내기 위해 수사에 쓰이는 과학적 수단이나 방법, 기술 등을 말하며 '범죄 과학'으 로도 부른다. 디지털 포렌식은 어떤 수사를 말하는 것일 까? |  |
| 탐구학습 | 정보보안 분야의 세부직업 정보보안 분야 전문가들의 역할 | ∘정보보안 분야는 어떤 세부 직업들이 있을까요? ∘정보보안 분야 전문가들은 어떤 일을 할까요? |  |
| [10분] 쉬는 시간 (쉬는 시간은 10분으로 하고, 45분에서 55분 사이에 시작해주세요.) |  |  |  |
| 창의적 체험 학습 | [20분] 프로그래밍 언어 체험 [20분] 포렌식 툴 | ∘화면에 자신의 이름을 출력하는 간단한 C 언어 프로그램 을 작성하고 실행하면서 프로그래밍 과정에서 하게 되 는 edit, compile, run, debugging 등의 용어를 이해한 다. ∘포렌식툴을 이용하여 이미징을 실행한다. 원본의 훼손 방 지를 위하여 동일한 해쉬값의 사본을 생성한다. |  |
| 생각 정리 하기 | [5분] 활동지 정리 [5분] 설문조사 | - 이번 차시 학습 내용 정리 - (생각정리하기) 완성하기 및 정답 확인 |  |

# TTPs_2_스피어_피싱으로_정보를_수집하는_공격망_구성_방식.pdf

   …

칸이 모자란 줄 : 0개


### 정리

| | |
| --- | --- |
| 파싱한 문서 | 5건 (hwpx 2 · pdf 3) — 실패 0 |
| 블록 | 법령은 천 개 단위, 보고서는 열 몇 개 |
| 파서가 잡은 표 | 2개 — 둘 다 제목 상자 |
| `pdfplumber` 가 찾은 표 | 33개 → 거르고 9개 |
| 저장 | `documents_tables.md` |

**문서의 종류가 파서에 무엇을 요구하는지를 정한다.** 사내 규정은 표에 금액·기한이 있어 표 추출이 핵심이었는데, 법령은 조문이 전부 글이고 표가 장식이다. 표가 정말로 있던 곳은 PDF 보고서였고, 그건 `parse_pdf` 가 못 보는 자리였다.

`parse_pdf` 에 `pdfplumber` 를 넣을지는 따로 정할 일이다. 시간이 십수 초씩 들고([[41_기술_스택_지도]]의 "무엇이 하나 더 늘어나는가"), 찾은 것의 4분의 3이 빈 상자다.